[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-08-fastapi-integration.ipynb#scrollTo=aa000001)

---
# Day 8 · FastAPI Integration — Request and Response Models
**certified-journeys / pydantic-certified** · Practice · API Schema Design

> **Goal for today:** Wire Pydantic models into a FastAPI app as typed request bodies and response schemas, use `response_model_exclude_unset`, add OpenAPI examples, and validate query parameters — all tested in-notebook with `TestClient`.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings fastapi httpx


## Step 1 · Request and Response Models

FastAPI uses Pydantic models directly as the schema for request bodies and responses.
The golden rule: **never expose your internal model directly** — keep separate
`Request` and `Response` models.

- `CreateUserRequest` — defines what the client must send
- `UserResponse` — defines what the API exposes (no password hash, no internal IDs)

**Docs:** https://fastapi.tiangolo.com/tutorial/body/


In [ ]:
from __future__ import annotations
import hashlib
import uuid
from datetime import datetime
from typing import Optional

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_validator


# ── Request model: what the client sends ──────────────────────────────────────
class CreateUserRequest(BaseModel):
    username: str = Field(
        min_length=3, max_length=32,
        examples=["alice"],
        description="Unique username, 3–32 characters",
    )
    email: str = Field(
        examples=["alice@example.com"],
        description="User email address",
    )
    password: str = Field(
        min_length=8,
        examples=["s3cur3P@ss!"],
        description="Password, minimum 8 characters",
    )
    full_name: Optional[str] = Field(
        default=None,
        examples=["Alice Smith"],
    )

    @field_validator("username")
    @classmethod
    def username_alphanumeric(cls, v: str) -> str:
        if not v.replace("_", "").replace("-", "").isalnum():
            raise ValueError("Username may only contain letters, digits, - and _")
        return v.lower()


# ── Response model: what the API returns (no password) ───────────────────────
class UserResponse(BaseModel):
    user_id: str
    username: str
    email: str
    full_name: Optional[str] = None
    created_at: datetime
    is_active: bool = True


# ── In-memory user store (production would use a database) ───────────────────
users_db: dict[str, dict] = {}


app = FastAPI(title="User API", version="1.0.0")


@app.post(
    "/users",
    response_model=UserResponse,         # FastAPI filters/validates output
    status_code=201,
)
def create_user(payload: CreateUserRequest) -> UserResponse:
    if payload.username in users_db:
        raise HTTPException(
            status_code=409,
            detail={"error": "username_taken", "username": payload.username},
        )

    # Hash password — never store plaintext
    pw_hash = hashlib.sha256(payload.password.encode()).hexdigest()
    user = {
        "user_id": str(uuid.uuid4()),
        "username": payload.username,
        "email": payload.email,
        "full_name": payload.full_name,
        "password_hash": pw_hash,       # stored internally, NOT in UserResponse
        "created_at": datetime.utcnow(),
        "is_active": True,
    }
    users_db[payload.username] = user
    return UserResponse(**user)


# ── Test it ───────────────────────────────────────────────────────────────────
client = TestClient(app)

resp = client.post("/users", json={
    "username": "alice",
    "email": "alice@example.com",
    "password": "s3cur3P@ss!",
    "full_name": "Alice Smith",
})
print(f"Status: {resp.status_code}")
body = resp.json()
print(f"Response keys: {sorted(body.keys())}")
print(f"password_hash in response: {'password_hash' in body}")
print(f"\nFull response:\n{body}")


### What just happened?

- **`response_model=UserResponse`** is the API contract: FastAPI filters the returned data through `UserResponse`, so `password_hash` is silently excluded from every response.
- **`Field(examples=[...])`** populates the OpenAPI schema — those values appear in Swagger UI automatically.
- **`HTTPException` with a dict `detail`** sends structured JSON errors that clients can parse programmatically.
- `TestClient` runs the ASGI app in-process — no server needed, perfect for notebooks.


## Step 2 · `response_model_exclude_unset`

When a response model has optional fields with defaults, FastAPI by default
serializes **all fields**, including unset ones. Use `response_model_exclude_unset=True`
to return only fields the endpoint explicitly set.

This is important for partial-update (PATCH) endpoints and sparse JSON APIs.

**Docs:** https://fastapi.tiangolo.com/tutorial/response-model/#use-the-response_model_exclude_unset-parameter


In [ ]:
from typing import Optional
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field


class ProductResponse(BaseModel):
    product_id: str
    name: str
    price: float
    description: Optional[str] = None   # optional — may or may not be set
    stock_count: Optional[int] = None   # optional — may or may not be set
    sku: Optional[str] = None


products_app = FastAPI()


@products_app.get(
    "/products/{product_id}",
    response_model=ProductResponse,
    response_model_exclude_unset=True,   # only return fields we actually set
)
def get_product(product_id: str):
    # Simulate a DB record that only has partial data
    return ProductResponse(
        product_id=product_id,
        name="Acme Widget",
        price=9.99,
        # description, stock_count, sku are NOT set — they will be excluded
    )


@products_app.get(
    "/products/{product_id}/full",
    response_model=ProductResponse,
    # No exclude_unset — all fields including None defaults are returned
)
def get_product_full(product_id: str):
    return ProductResponse(
        product_id=product_id,
        name="Acme Widget",
        price=9.99,
    )


pc = TestClient(products_app)

sparse = pc.get("/products/P001").json()
full   = pc.get("/products/P001/full").json()

print("With response_model_exclude_unset=True:")
print(f"  keys: {sorted(sparse.keys())}")
print(f"  body: {sparse}")

print("\nWithout (all fields, including None):")
print(f"  keys: {sorted(full.keys())}")
print(f"  body: {full}")


### What just happened?

- **Without `exclude_unset`**: all 6 fields appear in the response, with `None` for unset optionals — noisy and potentially misleading.
- **With `exclude_unset=True`**: only `product_id`, `name`, `price` are returned — exactly what was set.
- This distinction matters for PATCH responses (clients must not confuse "field was set to null" from "field was not touched").
- Internally, Pydantic tracks `model_fields_set` — the set of field names explicitly provided at construction time.


## Step 3 · JSON Schema Preview with `model_json_schema()`

`model_json_schema()` generates the full JSON Schema for a model.
FastAPI calls this internally to build the OpenAPI spec.
You can call it yourself to inspect, share, or validate against external tools.

**Docs:** https://docs.pydantic.dev/latest/concepts/json_schema/


In [ ]:
import json

schema = CreateUserRequest.model_json_schema()
print(json.dumps(schema, indent=2))


### What just happened?

- **`model_json_schema()`** returns the JSON Schema Draft 7 representation used by OpenAPI.
- `Field(examples=[...])` values appear under the `"examples"` key in each property.
- `min_length`, `max_length`, and other constraints translate to JSON Schema keywords (`minLength`, `maxLength`).
- You can share this schema with frontend teams or use it to configure validation in other systems.


## Step 4 · Validating Query Parameters with `model_validate`

FastAPI can declare query parameters individually, but for complex filtering
you can collect them into a Pydantic model using a **dependency**.
The dependency validates the raw query dict and injects a typed model.

**Docs:** https://fastapi.tiangolo.com/tutorial/body/


In [ ]:
from typing import Optional
from fastapi import Depends, FastAPI, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, model_validator


class UserSearchParams(BaseModel):
    """Query parameters for the user search endpoint."""
    q: Optional[str] = Field(default=None, description="Search query string")
    min_age: Optional[int] = Field(default=None, ge=0, le=150)
    max_age: Optional[int] = Field(default=None, ge=0, le=150)
    is_active: bool = True
    page: int = Field(default=1, ge=1)
    page_size: int = Field(default=20, ge=1, le=100)

    @model_validator(mode="after")
    def validate_age_range(self) -> "UserSearchParams":
        if self.min_age is not None and self.max_age is not None:
            if self.min_age > self.max_age:
                raise ValueError("min_age must be <= max_age")
        return self


def search_params_dep(
    q: Optional[str] = Query(default=None),
    min_age: Optional[int] = Query(default=None),
    max_age: Optional[int] = Query(default=None),
    is_active: bool = Query(default=True),
    page: int = Query(default=1),
    page_size: int = Query(default=20),
) -> UserSearchParams:
    """Dependency: collects raw query params and validates via Pydantic model."""
    return UserSearchParams.model_validate({
        "q": q, "min_age": min_age, "max_age": max_age,
        "is_active": is_active, "page": page, "page_size": page_size,
    })


search_app = FastAPI()


@search_app.get("/users/search")
def search_users(params: UserSearchParams = Depends(search_params_dep)):
    return {
        "query": params.q,
        "filters": {"min_age": params.min_age, "max_age": params.max_age},
        "page": params.page,
        "page_size": params.page_size,
        "results": [],  # real app would query the database
    }


sc = TestClient(search_app)

# Valid search
resp = sc.get("/users/search", params={"q": "alice", "min_age": "18", "max_age": "65", "page": "2"})
print(f"Valid search: {resp.status_code}")
print(f"Body: {resp.json()}")

# Invalid: min_age > max_age (cross-field validator)
resp2 = sc.get("/users/search", params={"min_age": "50", "max_age": "30"})
print(f"\nInvalid age range: {resp2.status_code}")
print(f"Error: {resp2.json()}")


### What just happened?

- **Dependency injection** with `Depends(search_params_dep)` lets us collect scattered query params into one validated Pydantic model.
- **`model_validator(mode='after')`** runs after all fields are parsed — ideal for cross-field constraints like `min_age <= max_age`.
- Cross-field violations surface as **422 Unprocessable Entity** responses — FastAPI's standard validation error code.
- This pattern also works for headers, cookies, and path parameters via `Header()`, `Cookie()`, and `Path()`.


## Step 5 · `Field(examples=[...])` and OpenAPI Schema

Examples set on `Field` populate the OpenAPI spec so they appear in Swagger UI
automatically. You can also add `json_schema_extra` to the model `Config` for
model-level examples.

**Docs:** https://fastapi.tiangolo.com/tutorial/schema-extra-example/


In [ ]:
import json
from pydantic import BaseModel, Field
from fastapi import FastAPI
from fastapi.testclient import TestClient


class CreateOrderRequest(BaseModel):
    customer_id: str = Field(
        examples=["cust-001"],
        description="Customer identifier",
    )
    items: list[dict] = Field(
        examples=[[{"sku": "WIDGET-A", "qty": 2}, {"sku": "WIDGET-B", "qty": 1}]],
        description="List of order items with sku and qty",
    )
    coupon_code: str | None = Field(
        default=None,
        examples=["SAVE10", None],
    )

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "customer_id": "cust-001",
                    "items": [{"sku": "WIDGET-A", "qty": 2}],
                    "coupon_code": "SAVE10",
                }
            ]
        }
    }


order_app = FastAPI(title="Order API")


@order_app.post("/orders")
def create_order(payload: CreateOrderRequest):
    return {"order_id": "ORD-999", "customer_id": payload.customer_id}


oc = TestClient(order_app)

# Inspect the OpenAPI schema to confirm examples are embedded
openapi = oc.get("/openapi.json").json()
order_schema = openapi["components"]["schemas"]["CreateOrderRequest"]

print("OpenAPI schema for CreateOrderRequest:")
print(json.dumps(order_schema, indent=2))


### What just happened?

- **`Field(examples=[...])`** at the property level and **`json_schema_extra`** at the model level both feed the OpenAPI spec.
- `GET /openapi.json` returns the live spec; `TestClient` makes it easy to inspect.
- The `examples` array on the model appears under the component schema and is picked up by Swagger UI as pre-filled request bodies.
- Multiple examples per field are supported — each appears as a selectable option in the UI.


## Step 6 · GET Endpoint with Response Model

Response models work on `GET` endpoints too. They act as the serialization
and filtering layer — even if your internal representation has extra fields,
only the response model fields are sent to the client.


In [ ]:
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel


class UserPublicResponse(BaseModel):
    user_id: str
    username: str
    full_name: Optional[str] = None
    is_active: bool
    # Note: email and password_hash are intentionally excluded


# Simulate internal DB rows (contain private fields)
_USERS = {
    "u1": {"user_id": "u1", "username": "alice", "full_name": "Alice Smith",
           "email": "alice@example.com", "password_hash": "abc123", "is_active": True},
    "u2": {"user_id": "u2", "username": "bob",   "full_name": None,
           "email": "bob@example.com",   "password_hash": "def456", "is_active": False},
}


users_list_app = FastAPI()


@users_list_app.get("/users", response_model=List[UserPublicResponse])
def list_users():
    return list(_USERS.values())  # dicts are coerced by response_model


@users_list_app.get("/users/{user_id}", response_model=UserPublicResponse)
def get_user(user_id: str):
    user = _USERS.get(user_id)
    if not user:
        raise HTTPException(status_code=404, detail={"error": "not_found", "user_id": user_id})
    return user


lc = TestClient(users_list_app)

users = lc.get("/users").json()
print("List response:")
for u in users:
    print(f"  {u}")
    print(f"  'email' leaked: {'email' in u}, 'password_hash' leaked: {'password_hash' in u}")

print()
single = lc.get("/users/u1").json()
print(f"Single user keys: {sorted(single.keys())}")

missing = lc.get("/users/u999")
print(f"\nMissing user: {missing.status_code} → {missing.json()}")


### What just happened?

- FastAPI's `response_model` serializes **dicts** just as well as model instances — it validates the return value through the response model regardless.
- `email` and `password_hash` are silently stripped from each user in the list — the contract is enforced automatically.
- **`List[UserPublicResponse]`** as a response model applies the same filtering to every item in the list.
- Structured `HTTPException` `detail` (a dict) produces a JSON body that clients can parse programmatically.


## Step 7 · Putting It Together — Full CRUD Slice

A complete create → read → error flow demonstrating all Day 8 patterns together.


In [ ]:
import json
from typing import Optional
import uuid
from datetime import datetime

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator


class ItemCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100, examples=["Widget A"])
    price: float = Field(gt=0, examples=[9.99])
    tags: list[str] = Field(default_factory=list, examples=[["electronics", "sale"]])

    @field_validator("tags")
    @classmethod
    def tags_lowercase(cls, v: list[str]) -> list[str]:
        return [t.lower().strip() for t in v]


class ItemResponse(BaseModel):
    item_id: str
    name: str
    price: float
    tags: list[str]
    created_at: datetime


item_app = FastAPI(title="Item API")
_items: dict[str, dict] = {}


@item_app.post("/items", response_model=ItemResponse, status_code=201)
def create_item(payload: ItemCreate) -> dict:
    item = {
        "item_id": str(uuid.uuid4())[:8],
        "name": payload.name,
        "price": payload.price,
        "tags": payload.tags,
        "created_at": datetime.utcnow(),
    }
    _items[item["item_id"]] = item
    return item


@item_app.get("/items/{item_id}", response_model=ItemResponse)
def get_item(item_id: str) -> dict:
    if item_id not in _items:
        raise HTTPException(404, detail={"error": "not_found", "item_id": item_id})
    return _items[item_id]


ic = TestClient(item_app)

# Create
r1 = ic.post("/items", json={"name": "Widget A", "price": 9.99, "tags": ["Electronics", "SALE"]})
print(f"Create: {r1.status_code}")
created = r1.json()
print(f"Tags normalized: {created['tags']}")

# Read back
r2 = ic.get(f"/items/{created['item_id']}")
print(f"Get: {r2.status_code} → {r2.json()['name']}")

# 422 validation error
r3 = ic.post("/items", json={"name": "", "price": -1})
print(f"\nValidation error: {r3.status_code}")
print(json.dumps(r3.json(), indent=2))


### What just happened?

- **Field validators run before storage**: tags were normalized to lowercase before being saved.
- **422 with structured `detail`**: FastAPI returns Pydantic's validation errors in a consistent JSON format — each error has `loc`, `msg`, and `type`.
- **`response_model` coerces dicts**: returning a plain dict from an endpoint is fine — FastAPI validates it through the response model.
- This full slice (create → validate → store → retrieve) covers the complete CRUD pattern with Pydantic+FastAPI.


In [ ]:
# Challenge: Build a Blog Post API slice
#
# Requirements:
#   1. CreatePostRequest:
#        title: str (1–200 chars), body: str (min 10 chars),
#        tags: list[str] (max 5 tags), published: bool = False
#      Add a validator that ensures title doesn't contain HTML tags (reject if '<' in title)
#
#   2. PostResponse:
#        post_id, title, body, tags, published, created_at, word_count (computed)
#
#   3. Endpoints:
#        POST /posts  → 201 with PostResponse
#        GET  /posts  → List[PostResponse] (filter by ?published=true/false if provided)
#
#   4. Test with TestClient:
#        - Create 2 posts (one published, one not)
#        - List all, then filter by published=true
#        - Verify word_count is correct
#        - Verify HTML title is rejected with 422
#

# Your solution here
# from fastapi import FastAPI
# from fastapi.testclient import TestClient
# ...


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| Separate Request/Response models | Never expose internal models; `response_model` is your API contract |
| `response_model_exclude_unset=True` | Returns only explicitly-set fields — vital for PATCH endpoints |
| `Field(examples=[...])` | Populates Swagger UI automatically — free documentation |
| `model_json_schema()` | Inspect or share the full JSON Schema for any model |
| Query param dependency | Collect and validate query params with `model_validate` inside a `Depends()` |
| `HTTPException` with dict detail | Structured errors that clients can parse programmatically |
| `TestClient` | Run FastAPI in-process — no server needed, works in notebooks |
| 422 Unprocessable Entity | FastAPI's standard response for Pydantic validation failures |

> **Tip:** Use separate Request and Response models — never expose your internal ORM model directly. The `response_model` parameter is your API contract.

---
## What's next
**Day 9** → Performance patterns and Pydantic internals — `ConfigDict(frozen=True)`, `TypeAdapter`, `model_construct()`, and benchmarking `model_validate_json()` vs `json.loads()`.

Mark Day 8 complete in your [tracker](../index.html).
